# 02 · Experiments

3 representations × 3 architectures, each evaluated with k-fold cross-validation.
Every run saves its fold scores to `results/<architecture>__<representation>.json`.

In [9]:
# Setup: make the course code (part1) and our code (src) importable.
# Run notebooks from this folder (JupyterLab / VS Code do this by default).
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
from part1 import experiment
from src import config, data, encoders, results_io

## Data and folds

In [11]:
LABEL_MODE = "fine"   # "fine" (28) | "ekman" (7) | "sentiment" (4)
pairs, outcome_space, stats = data.load_pairs(label_mode=LABEL_MODE)
print(stats)
folds = experiment.cross_validation_split(pairs, k=config.K_FOLDS, seed=config.SEED)

{'label_mode': 'fine', 'min_raters': 2, 'comments_with_label': 54263, 'kept': 45446, 'dropped_multilabel': 8817}


## Tokeniser

Reuse the tutorial's or train our own? On what corpus, which vocab size?
Note the choice + reason in the decisions log.

In [ ]:
corpus = data.tokenizer_corpus()
tokenizer = experiment.train_tokenizer(corpus, vocab_size=58800, min_frequency=1)
tokenizer.get_vocab_size()

## Representations

In [13]:
representations = {
    "indicator": encoders.TokenIndicator(tokenizer),
    "bow": encoders.BagOfWords(tokenizer),
    "tfidf": encoders.TfIdf(tokenizer),
}

## Log-linear

In [14]:
for rep_name, enc in representations.items():
    name = f"loglin__{rep_name}"
    out = experiment.cross_validation_experiment(
        folds, outcome_space, enc, "logistic", save_to=str(config.MODELS_DIR / name)
    )
    results_io.save_cv(name, out, vocab_size=tokenizer.get_vocab_size())
    print(name, out["accuracies"])

  0%|          | 0/5 [00:00<?, ?it/s]

/Users/nordin/Documents/UvA/Master/NLP/nlp1-rosa-nordin/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/nordin/Documents/UvA/Master/NLP/nlp1-rosa-nordin/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/nordin/Documents/UvA/Master/NLP/nlp1-rosa-nordin/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/nordin/Documents/UvA/Master/NLP/nlp1-rosa-nordin/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter 

loglin__indicator OrderedDict({0: 0.5382838283828383, 1: 0.5539663329299153, 2: 0.5384530751457807, 3: 0.5440642534932336, 4: 0.5375728903069644})


  0%|          | 0/5 [00:00<?, ?it/s]

/Users/nordin/Documents/UvA/Master/NLP/nlp1-rosa-nordin/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/nordin/Documents/UvA/Master/NLP/nlp1-rosa-nordin/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/nordin/Documents/UvA/Master/NLP/nlp1-rosa-nordin/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/nordin/Documents/UvA/Master/NLP/nlp1-rosa-nordin/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter 

loglin__bow OrderedDict({0: 0.5378437843784378, 1: 0.5509957090989108, 2: 0.53460226647596, 3: 0.5425239300253053, 4: 0.5392232368797447})


## MLP (one hidden layer)

In [15]:
smlp_args = dict(hidden_dim=128, dropout=0.1, num_epochs=10)  # TODO: tune

for rep_name, enc in representations.items():
    name = f"mlp__{rep_name}"
    out = experiment.cross_validation_experiment(
        folds, outcome_space, enc, "mlp", save_to=str(config.MODELS_DIR / name), smlp_args=smlp_args
    )
    results_io.save_cv(name, out, vocab_size=tokenizer.get_vocab_size(), **smlp_args)
    print(name, out["accuracies"])

  0%|          | 0/5 [00:00<?, ?it/s]

Transforming documents...
Document transformation done! Time taken: 4.009 seconds
Dataloader done! Time taken: 27.855 seconds
Training...


  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 2.2731 | train acc: 0.4329 | val loss: 1.8570 | val acc: 0.5046
Epoch 02 | train loss: 1.3179 | train acc: 0.6241 | val loss: 1.6871 | val acc: 0.5454
Epoch 03 | train loss: 0.7047 | train acc: 0.8015 | val loss: 1.7763 | val acc: 0.5332
Epoch 04 | train loss: 0.3862 | train acc: 0.8971 | val loss: 1.9334 | val acc: 0.5199
Epoch 05 | train loss: 0.2305 | train acc: 0.9432 | val loss: 2.1280 | val acc: 0.5175
Epoch 06 | train loss: 0.1510 | train acc: 0.9645 | val loss: 2.2996 | val acc: 0.5113
Epoch 07 | train loss: 0.1047 | train acc: 0.9759 | val loss: 2.4930 | val acc: 0.5041
Epoch 08 | train loss: 0.0764 | train acc: 0.9825 | val loss: 2.6624 | val acc: 0.4981
Epoch 09 | train loss: 0.0580 | train acc: 0.9876 | val loss: 2.8200 | val acc: 0.4948
Epoch 10 | train loss: 0.0446 | train acc: 0.9905 | val loss: 3.0109 | val acc: 0.4870
Training done! Time taken: 79.284 seconds
Transforming documents...
Document transformation done! Time taken: 4.293 seconds
Datalo

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 2.2792 | train acc: 0.4286 | val loss: 1.8082 | val acc: 0.5205
Epoch 02 | train loss: 1.3106 | train acc: 0.6243 | val loss: 1.6367 | val acc: 0.5570
Epoch 03 | train loss: 0.6949 | train acc: 0.8012 | val loss: 1.7284 | val acc: 0.5434
Epoch 04 | train loss: 0.3818 | train acc: 0.8989 | val loss: 1.8884 | val acc: 0.5325
Epoch 05 | train loss: 0.2313 | train acc: 0.9431 | val loss: 2.0396 | val acc: 0.5225
Epoch 06 | train loss: 0.1530 | train acc: 0.9628 | val loss: 2.2045 | val acc: 0.5111
Epoch 07 | train loss: 0.1073 | train acc: 0.9752 | val loss: 2.3799 | val acc: 0.5041
Epoch 08 | train loss: 0.0780 | train acc: 0.9823 | val loss: 2.5617 | val acc: 0.5027
Epoch 09 | train loss: 0.0611 | train acc: 0.9859 | val loss: 2.7275 | val acc: 0.4981


## Our third architecture

See `src/models.py`.

In [ ]:
# TODO